In [0]:
data = spark.sql("""
    select * from (values
        (1, 'Credit card'),
        (2, 'Cash'),
        (3, 'No charge'),
        (4, 'Dispute')
    ) as t(payment_id, payment_type)
""")
data.createOrReplaceTempView("payment_dim")
data.display()

isi= spark.sql("select * from payment_dim")
isi.show()

import os 
data= os.getcwd()
print(os.listdir(data))

payment_id,payment_type
1,Credit card
2,Cash
3,No charge
4,Dispute


+----------+------------+
|payment_id|payment_type|
+----------+------------+
|         1| Credit card|
|         2|        Cash|
|         3|   No charge|
|         4|     Dispute|
+----------+------------+

['New Notebook 2026-08-18 00:51:11.ipynb', 'New Notebook 2026-08-18 11:00:12.ipynb', 'yellow_taxi_hourly_summary.parquet', 'yellow_tripdata_2024-02.parquet', 'gold_peak_hours']


In [0]:
import os
data = os.getcwd()
uni = os.path.join(data, "yellow_tripdata_2024-02.parquet")

if os.path.exists(uni):
    brz = spark.read.parquet(uni)
    brz.createOrReplaceTempView("yellow_taxi_trips")
    
    df = spark.sql("""
        WITH temp AS (
            SELECT
                vendorID,
                CAST(tpep_pickup_datetime AS DATE) AS pickup_date,
                trip_distance,
                fare_amount,
                pd.payment_type,
                CASE
                    WHEN trip_distance > 100 AND fare_amount < 10 THEN 'yes'
                    ELSE 'no'
                END AS is_suspicious,
                -- BONUS FIX: Changed to partition by the DATE, not the exact second!
                row_number() OVER (PARTITION BY CAST(tpep_pickup_datetime AS DATE) ORDER BY fare_amount DESC) AS rank
            FROM yellow_taxi_trips AS yl
            JOIN payment_dim AS pd
              ON yl.payment_type = pd.payment_id
            WHERE  
                tpep_pickup_datetime >= '2024-02-01' AND tpep_pickup_datetime < '2024-03-01'
        )
        SELECT pickup_date, payment_type, fare_amount, is_suspicious
        FROM temp
        WHERE rank <= 2
    """)
    
    print("Saving as a Managed Delta Table...")
    # THE FIX: Removed the file path and used saveAsTable() instead!
    df.write.format("delta").mode("overwrite").partitionBy("pickup_date").saveAsTable("suspicious_fare")
    
    print("Success! Table saved safely inside Databricks.")
else:
    print("file does not exist")

Saving as a Managed Delta Table...
Success! Table saved safely inside Databricks.


In [0]:
# from pyspark.sql import Row
# from datetime import date

display(spark.sql("select * from suspicious_fare"))

updates_data = [
    Row(pickup_date="2024-02-05", payment_type="Credit card", fare_amount=8.50, is_suspicious='no'),
    Row(pickup_date="2024-02-29", payment_type="Credit card", fare_amount=150.00, is_suspicious='yes')
]

updates_df = spark.createDataFrame(updates_data)
updates_df.createOrReplaceTempView("fare_updates")

spark.sql("""
    MERGE INTO suspicious_fare AS source 
    USING fare_updates AS updates
    ON source.pickup_date = updates.pickup_date AND source.payment_type = updates.payment_type
    WHEN MATCHED THEN 
      UPDATE SET source.fare_amount = updates.fare_amount, source.is_suspicious = updates.is_suspicious
    WHEN NOT MATCHED THEN
      INSERT (pickup_date, payment_type, fare_amount, is_suspicious)
      VALUES (updates.pickup_date, updates.payment_type, updates.fare_amount, updates.is_suspicious)
""")

display(spark.sql("select * from suspicious_fare"))

pickup_date,payment_type,fare_amount,is_suspicious
2024-02-05,Credit card,495.0,no
2024-02-05,Credit card,480.0,no
2024-02-09,Credit card,400.0,no
2024-02-09,Credit card,378.0,no
2024-02-07,Credit card,495.0,no
2024-02-07,Credit card,495.0,no
2024-02-11,No charge,600.0,no
2024-02-11,No charge,587.5,no
2024-02-23,No charge,403.4,no
2024-02-23,Credit card,391.5,no


pickup_date,payment_type,fare_amount,is_suspicious
2024-02-05,Credit card,8.5,no
2024-02-05,Credit card,8.5,no
2024-02-09,Credit card,400.0,no
2024-02-09,Credit card,378.0,no
2024-02-07,Credit card,495.0,no
2024-02-07,Credit card,495.0,no
2024-02-11,No charge,600.0,no
2024-02-11,No charge,587.5,no
2024-02-23,No charge,403.4,no
2024-02-23,Credit card,391.5,no
